# Results for RQ1 

## Setup

* To replicate RQ1's evaluation use the default ```src/config.py``` values
* Follow the instructions on ```Readme.md```
* Inside the *pipeline* folder, run ```python -m src.main``` 

## Efficiency

In [ ]:
from src.config import *

import json
import pandas as pd

def _load_clones(filename):
    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)
    clones = []
    for entry in data:
        entry_id = entry.get("id", "unknown")
        for clone in entry.get("clones", []):
            clones.append({
                "entry_id": entry_id,
                "model": clone.get("model", "unknown"),
                "context": clone.get("context", "unknown"),
                "refac": clone.get("refacs", "unknown"),
                "strategy": clone.get("strategy", "unknown"),
                "clone_id": clone.get("clone_id", None),
            })
    return pd.DataFrame(clones)

# === Load all stages ===
df0 = _load_clones(OUT_PATH)
df1 = _load_clones(FILTERED_PATH_CODEBLEU)
df2 = _load_clones(FILTERED_PATH_TESTS)
df3 = _load_clones(REPROMPT_PATH)
df5 = _load_clones(FINAL_DATASET)

# Note: df4 (clones that pass all tests) is implicitly represented inside df2 and df3

# === Count clones per configuration at each stage ===
def _count_stage(df, label):
    return (
        df.groupby(["model", "context", "refac", "strategy"])
          .size()
          .reset_index(name=f"N_{label}")
    )

counts = [
    _count_stage(df0, "0"),
    _count_stage(df1, "1"),
    _count_stage(df2, "2"),
    _count_stage(df3, "3"),
    _count_stage(df5, "5"),
]

eff_df = counts[0]
for c in counts[1:]:
    eff_df = eff_df.merge(c, on=["model", "context", "refac", "strategy"], how="left")

eff_df = eff_df.fillna(0)

# === Compute efficiency and survival ratios ===
eff_df["efficiency"] = eff_df["N_5"] / eff_df["N_0"]

# Survival at each stage
eff_df["S_codebleu"] = eff_df["N_1"] / eff_df["N_0"]
eff_df["S_tests"] = eff_df["N_2"] / eff_df["N_1"]
eff_df["S_reprompt"] = eff_df["N_3"] / eff_df["N_2"]
eff_df["S_final"] = eff_df["N_5"] / (eff_df["N_3"] + eff_df["N_2"])

eff_df = eff_df.round(4)
eff_df = eff_df.sort_values(by="efficiency", ascending=False)

# Save results
eff_df.to_csv("efficiency_summary.csv", index=False)
print(eff_df.head(10))

